# Milestone 2: Qdrant Vector Search & Retrieval Debugging

This notebook verifies and inspects vector similarity search on the embedded Qdrant vector store (`fitness_docs`).

We evaluate:
1. Collection status and indexed points count.
2. Cosine similarity scores across the 3 fitness knowledge pillars.
3. Metadata filtering (e.g. scoping search strictly to `nutrition` or `health_issues`).
4. Top-K ranking inspection for grounding.

In [ ]:
import sys
from pathlib import Path

# Add src to Python path
root_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(root_dir / "src"))

from rag_fitness_coach.vectorstore.client import get_collection_info, get_qdrant_client
from rag_fitness_coach.vectorstore.indexer import similarity_search
import pandas as pd

## 1. Inspect Qdrant Collection Statistics

In [ ]:
info = get_collection_info()
print(f"Collection: {info['name']}")
print(f"Status: {info['status']}")
print(f"Total vectors: {info['vectors_count']}")

## 2. Test Multi-Domain Fitness Queries

In [ ]:
test_queries = [
    ("How many sets per week maximize muscle hypertrophy?", None),
    ("What are safe exercises for chronic low back pain rehab?", "health_issues"),
    ("How much protein is optimal per day for athletes?", "nutrition"),
    ("Cardiovascular and mitochondrial adaptations to aerobic exercise", "exercise_library"),
]

for query, filter_pillar in test_queries:
    print(f"\n{'='*75}\nQUERY: {query} (Filter: {filter_pillar})\n{'='*75}")
    results = similarity_search(query=query, top_k=3, pillar_filter=filter_pillar)
    for i, (doc, score) in enumerate(results, 1):
        title = doc.metadata.get('title')
        pillar = doc.metadata.get('pillar')
        preview = doc.page_content.strip()[:200].replace('\n', ' ')
        print(f"#{i} [Score: {score:.4f}] [{pillar}] {title}")
        print(f"   Preview: {preview}...")

## 3. Score Distribution Analysis

In [ ]:
sample_q = "resistance training progressive overload"
top_candidates = similarity_search(sample_q, top_k=10)

scores_df = pd.DataFrame([
    {
        "rank": idx + 1,
        "score": score,
        "title": doc.metadata.get("title"),
        "pillar": doc.metadata.get("pillar"),
        "chars": len(doc.page_content)
    }
    for idx, (doc, score) in enumerate(top_candidates)
])

scores_df